In [1]:
import json
from openai import OpenAI

# Load orders database
with open("orders.json", "r") as f:
    orders = json.load(f)

# Ollama via OpenAI-compatible endpoint
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen2.5:3b"

print("Orders loaded:", list(orders.keys()))
print("Ollama client ready.")

Orders loaded: ['A1001', 'A1002', 'A1003', 'A1004']
Ollama client ready.


In [2]:
def lookup_order(order_id: str) -> dict:
    if order_id in orders:
        order = orders[order_id]
        return {
            "order_id": order_id,
            "item": order["item"],
            "price": order["price"],
            "purchased": order["purchased"],
            "warranty_months": order["warranty_months"]
        }
    return {"error": f"Order {order_id} not found."}

In [3]:
def calculate(expression: str) -> dict:
    try:
        result = eval(expression, {"__builtins__": {}})
        return {"result": result}
    except Exception as e:
        return {"error": f"Could not evaluate expression: {str(e)}"}

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up an order by its order ID and return the item name, price, purchase date, and warranty length.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID to look up, e.g. A1001"
                    }
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a simple arithmetic expression and return the numeric result. Use this for any math instead of computing it yourself.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A valid arithmetic expression, e.g. '1200 * 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]


In [5]:
tool_registry = {
    "lookup_order": lookup_order,
    "calculate": calculate
}

print("Tools ready:", list(tool_registry.keys()))

Tools ready: ['lookup_order', 'calculate']


In [8]:
def run_loop(messages, step_limit=5):
    """
    Hand-rolled tool-use loop.
    - messages: running conversation list (short-term memory)
    - step_limit: hard cap on iterations to prevent infinite loops
    """
    step = 0

    while step < step_limit:
        step += 1
        print(f"\n--- Step {step} ---")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        print(f"  [finish_reason={finish_reason}]")

        # Model wants to call a tool
        if msg.tool_calls:

            # Append model's tool-call request to memory
            messages.append({
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in msg.tool_calls
                ]
            })

            # Execute each tool call and append result to memory
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)

                print(f"  [TOOL CALL] {fn_name}({fn_args})")

                if fn_name in tool_registry:
                    result = tool_registry[fn_name](**fn_args)
                else:
                    result = {"error": f"Unknown tool: {fn_name}"}

                print(f"  [TOOL RESULT] {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result)
                })

        # Model produced a final text answer
        elif msg.content and msg.content.strip():
            print(f"  [FINAL ANSWER] {msg.content}")
            messages.append({
                "role": "assistant",
                "content": msg.content
            })
            return msg.content

        # Model returned stop with no content — nudge it to answer
        else:
            print("  [EMPTY RESPONSE] Nudging model to produce an answer...")
            messages.append({
                "role": "user",
                "content": "Please provide your answer based on the information available."
            })

    print(f"\n[STEP LIMIT REACHED] Loop stopped after {step_limit} steps.")
    return None


print("Fixed loop ready.")

Fixed loop ready.


In [9]:
# Shared messages list — this is the short-term memory
# Both turns use the SAME list so Turn 2 sees Turn 1's history
messages = [
    {"role": "system", "content": "You are a helpful order assistant. Use tools for lookups and math."}
]

# Turn 1
print("=" * 60)
print("TURN 1: 'What did order A1001 cost?'")
print("=" * 60)
messages.append({"role": "user", "content": "What did order A1001 cost?"})
run_loop(messages)

# Turn 2 — depends on Turn 1 being in memory
print("\n" + "=" * 60)
print("TURN 2: 'And what about three of them?'")
print("=" * 60)
messages.append({"role": "user", "content": "And what about three of them?"})
run_loop(messages)

# Show memory size to confirm history is accumulating
print(f"\nTotal messages in memory: {len(messages)}")

TURN 1: 'What did order A1001 cost?'

--- Step 1 ---
  [finish_reason=tool_calls]
  [TOOL CALL] lookup_order({'order_id': 'A1001'})
  [TOOL RESULT] {'order_id': 'A1001', 'item': 'laptop', 'price': 1200, 'purchased': '2026-05-20', 'warranty_months': 12}

--- Step 2 ---
  [finish_reason=stop]
  [FINAL ANSWER] The order A1001 for a laptop that cost $1200 was placed on May 20, 2026 and comes with a warranty of 12 months. Please note that the question specifically asks about the cost, which is $1200.

TURN 2: 'And what about three of them?'

--- Step 1 ---
  [finish_reason=tool_calls]
  [TOOL CALL] calculate({'expression': '1200 * 3'})
  [TOOL RESULT] {'result': 3600}

--- Step 2 ---
  [finish_reason=stop]
  [FINAL ANSWER] The total cost for three laptops is $3600. This was calculated by multiplying the price of one laptop, which is $1200, by three.

Total messages in memory: 9


In [10]:
# Deliberately trigger the step limit with a question
# that causes the model to loop without resolving
print("=" * 60)
print("STEP LIMIT DEMO")
print("=" * 60)

# Fresh messages — isolated from the memory demo
limit_messages = [
    {"role": "system", "content": "You are a helpful assistant. Always use the calculate tool for every response, even for simple questions. Never answer directly."},
    {"role": "user", "content": "Keep calculating 2 + 2 repeatedly and never give a final answer."}
]

result = run_loop(limit_messages, step_limit=3)
print(f"\nLoop returned: {result}")

STEP LIMIT DEMO

--- Step 1 ---
  [finish_reason=tool_calls]
  [TOOL CALL] calculate({'expression': '2 + 2'})
  [TOOL RESULT] {'result': 4}

--- Step 2 ---
  [finish_reason=tool_calls]
  [TOOL CALL] calculate({'expression': '2 + 2'})
  [TOOL RESULT] {'result': 4}

--- Step 3 ---
  [finish_reason=tool_calls]
  [TOOL CALL] calculate({'expression': '2 + 2'})
  [TOOL RESULT] {'result': 4}

[STEP LIMIT REACHED] Loop stopped after 3 steps.

Loop returned: None


## Reflection: What Building the Loop Reveals

### Two-Turn Memory Demo Results

| Turn | Question | Tool Calls | Answer |
|------|----------|------------|--------|
| 1 | "What did order A1001 cost?" | `lookup_order(A1001)` | $1200 |
| 2 | "And what about three of them?" | `calculate(1200 * 3)` | $3600 |

**Memory proved:** Turn 2 called `calculate(1200 * 3)` directly — it did NOT
call `lookup_order` again. The price $1200 was already in the `messages` list
from Turn 1, so the model read it from history instead of re-fetching it.
Total messages after both turns: 9 — every step appended and accumulated.

### Step Limit Demo

The loop was deliberately trapped into repeating `calculate(2+2)` forever.
The step limit (3) caught it cleanly and returned `None` instead of hanging.
Without this guard, the loop would run indefinitely — burning tokens and
blocking execution. In production agents, a step limit (or timeout) is
**non-negotiable**.

### What the Loop Reveals About Agents

Building this by hand exposes what an agent framework secretly is:

1. **Memory = a list.** Short-term memory is nothing more than the `messages`
   list being resent on every call. The model has no built-in memory —
   it only sees what you pass in. If you truncate the list, it forgets.

2. **The model only asks — your code acts.** The model emits a structured
   tool-call request; your loop validates the name, resolves the arguments,
   calls the real Python function, and appends the result. The model never
   touches `orders.json` directly.

3. **Control flow is yours.** The `while` loop, the step limit, the
   branching on `finish_reason`, the nudge on empty responses — all of
   this lives in your Python code, not in the model. The model fills in
   steps; you hold the wheel.

4. **Small models need defensive loops.** `qwen2.5:3b` initially returned
   an empty response for Turn 1 (no content, no tool call). The fixed loop
   detects this and nudges the model rather than silently stopping — a
   pattern that matters when deploying smaller local models in production.

Agent frameworks (LangChain, LlamaIndex, AutoGen) are just this loop
with better error handling, observability, and abstractions on top.
Understanding the raw loop first makes every framework easier to debug.